In [ ]:
# Jupyter Notebook for eQTL study

In [ ]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import gc
import warnings
from itables import init_notebook_mode
import numba

warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

init_notebook_mode(all_interactive=True)

sc.settings.verbosity = 4
root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
script_dir = root_dir + 'scripts/'
results_dir = root_dir + 'results/'
data_dir = results_dir + '02PARSE/'
plate1_dir = data_dir + 'combine_plate1/all-sample/DGE_filtered/'
plate2_dir = data_dir + 'combine_plate1/all-sample/DGE_filtered/'
scanpy_dir = results_dir + '03SCANPY/'
sc.settings.figdir = results_dir + '/figs/'
sys.path.append(script_dir)  # Add the directory containing custom scripts to sys.path
os.environ['OMP_NUM_THREADS'] = '8'

# Import custom utility lists and functions
from anndata_utils import *
from gene_lists import *

# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(12,6), format='png')


In [ ]:
# Load data

In [ ]:
adata_p1 = sc.read(plate1_dir + 'anndata.h5ad', backed='r') 
adata_p2 = sc.read(plate2_dir + 'anndata.h5ad', backed='r') 

In [ ]:
# Add prefixes to cell names
adata_p1.obs_names = 'plate1_' + adata_p1.obs_names
adata_p2.obs_names = 'plate2_' + adata_p2.obs_names

# Concatenate the AnnData objects
adata = sc.concat([adata_p1, adata_p2])

# Rm 'hg38' from gene names
adata.var.index = adata.var.index.str.replace('_hg38', '')

adata

In [ ]:
adata.obs

In [ ]:
# QC metadata

In [ ]:
adata.obs['sample'] = adata.obs['sample'].str.replace('sample_', '')
adata.obs['plate'] = adata.obs.index.str.split('_').str[0]
adata = adata[~adata.obs['sample'].str.endswith(tuple(['WGE', 'Hipp', 'Thal']))]
adata.obs['sample'].value_counts()

In [ ]:
plt.hist(adata.var['n_cells_by_counts'], bins=500)
plt.xlabel('N cells expressing > 0')
plt.ylabel('log(N genes)') # for visual clarity
plt.axvline(2, color='red')
plt.yscale('log') 

sns.jointplot(
    data=adata.obs,
    x="log1p_total_counts",
    y="log1p_n_genes_by_counts",
    kind="hex",
)

In [ ]:
# Normalise

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# ID highly variable genes

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.25, n_jobs=8)
# This saves the original set of genes 
adata.raw = adata

adata = adata[:,adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)

In [ ]:
# PCA

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='') # scanpy generates the filename automatically

In [ ]:
# UMAP and Clustering

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50, n_jobs=8)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

In [ ]:
# Plotting

In [ ]:
# Helper function
sc.pl.umap(adata, color=['leiden'])

In [ ]:
# Violin plots
sc.pl.stacked_violin(adata, general_genes, groupby="leiden", row_palette=discreet_cols_n23, swap_axes=True)
#sc.pl.violin(adata, 'n_genes_by_counts', jitter=0.4, groupby = 'sample', rotation = 90, size=0)